<a href="https://colab.research.google.com/github/jadenshk/stellar-temperature/blob/main/ACST_Condensed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from scipy.optimize import curve_fit
import seaborn as sns

In [ ]:
!pip install astropy==5.3.1
from astropy.table import Table
from astropy.io import fits
from sklearn.model_selection import train_test_split

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 7.6 MB/s eta 0:00:00
  Attempting uninstall: astropy
    Found existing installation: astropy 5.3.4
    Uninstalling astropy-5.3.4:
      Successfully uninstalled astropy-5.3.4


In [ ]:
from patsy import dmatrix
import statsmodels.api as sm

In [ ]:
def get_teff(filename):

    data = open(filename)

    foundTeff = False
    teff_str = ""

    while foundTeff == False:
        line = data.readline()
        index = line.find("Teff")

        if index > -1:
            foundTeff = True
            teff_str = line

            break

    Teff = teff_str[83:(len(teff_str) - 3)]

    return Teff



In [ ]:
def get_data(file):

    file_image = file[1].data
    file_df = pd.DataFrame(file_image)
    file_df.columns = ["Wavelengths", "Fluxes", "Fluxes (normalized)"]
    return file_df

In [ ]:
def max(dtst):
    flx_max = -100000000
    n = -1
    n_max = n

    for i in dtst:
        n += 1
        if i > flx_max:
            flx_max = i
            n_max = n

    return (n_max, flx_max)

In [ ]:
def MM(lst, r_pm):

    k = 0
    width = 2 * r_pm + 1

    lst_sum = []
    lst_MM = []

    while k < len(lst):
        r_val = -r_pm   #r_val is the "distance" (so to speak) from the central index.
                        #This resets the r_val to be at the start value after all indeces within the range are added.
        sum = 0         #This also resets the sum of the values to zero after all indeces within the range are added.

        while r_val <= r_pm:
            if (r_val + k) >= 0 and (r_val + k) < len(lst):
                k_val = lst[k + r_val] #k_val is the value stored in the list at a specific index "k"
                sum += k_val

            r_val += 1

        lst_sum.append(sum)

        k += 1

    for i in lst_sum:
        j = i / width
        lst_MM.append(j)

    return (lst_MM)

In [ ]:
def temp(wvl_max):

    T = ((2.897771955 * (10 ** 7)) / wvl_max)

    return(T)

In [ ]:
#filename is the full path

DATA_PATH = "drive/MyDrive/Data for Research Project/"
OUTPUT_PATH = "drive/MyDrive/Research Project Results/"

def procedure(filename, range):

    teff = get_teff(DATA_PATH + filename)

    # Retrieve numerical data from file and making it into a pandas dataframe
    file = fits.open(DATA_PATH + filename)
    file_df = get_data(file)

## THIS IS FOR THE UNALTERED DATA ##

    # Plot the spectrum with original data
    file_df.plot(x = 'Wavelengths', y = 'Fluxes', kind = "line", title = 'Original Data').set_xticks([4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000])
    plt.savefig(OUTPUT_PATH + filename + '_original.png')
    plt.close()

    # Store the flux and wavelength values into lists
    file_flx = file_df["Fluxes"]
    file_wvl = file_df["Wavelengths"]

    # Find the max of the initial data and its corresponding wavelength
    file_index_max1, file_flx_max1 = max(file_flx)
    file_wvl_max1 = file_wvl[file_index_max1]

    # Find the temperature and store it
    temp1 = temp(file_wvl_max1)

    print("--unaltered data--")
    print("maximum flux: ..............", file_flx_max1)
    print("corresponging wavelength: ..", file_wvl_max1)
    print("temperature: ...............", temp1)

## THIS IS FOR THE CONVOLVED DATA ##

    # Create the convolved fluxes list and add to the dataframe
    file_flx_MM = MM(lst = file_flx, r_pm = range)
    file_df.insert(2, "Convolved Fluxes", file_flx_MM)

    # Plot the spectrum with convolved data
    file_df.plot(x = 'Wavelengths', y = 'Convolved Fluxes', kind = 'line', title = 'Convolved Data').set_xticks([4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000])
    plt.savefig(OUTPUT_PATH + filename + '_convolved.png')
    plt.close()

    # Find the max of the convolved data and its corresponding wavelength
    file_index_max2, file_flx_max2 = max(file_flx_MM)
    file_wvl_max2 = file_wvl[file_index_max2]

    # Find the temp and store it
    temp2 = temp(file_wvl_max2)

    print("--convoled data--")
    print("maximum flux: ..............", file_flx_max2)
    print("corresponging wavelength: ..", file_wvl_max2)
    print("temperature: ...............", temp2)

## THIS IS FOR THE CUBIC SPLINE REGRESSION ##

    # Find the location of the knots
        # this is done arbitrarily just based on the length of the list

    last_wvl = file_wvl[(len(file_wvl) - 1)]
    first_wvl = file_wvl[0]
    range_wvl = last_wvl - first_wvl

    knot1 = (1/8 * range_wvl) + first_wvl
    knot2 = (2/8 * range_wvl) + first_wvl
    knot3 = (3/8 * range_wvl) + first_wvl
    knot4 = (4/8 * range_wvl) + first_wvl
    knot5 = (5/8 * range_wvl) + first_wvl
    knot6 = (6/8 * range_wvl) + first_wvl
    knot7 = (7/8 * range_wvl) + first_wvl

    # Transform the data set to allow for the regression
    tr_file_wvl = dmatrix(
    "bs(train, knots=(knot1, knot2, knot3, knot4, knot5, knot6, knot7), degree=3, include_intercept=False)",
    {
        "train": file_wvl
    },

    return_type='dataframe'
    )

    # Run the actual curve-fitting function with the changed wavelength and ORIGINAL fluxes
    file_cs = sm.GLM(file_flx, tr_file_wvl).fit() # cs denotes "cubic spline"

    # Set up the graph to visualize data
    plt.figure(figsize=(10,10))
    xp = np.linspace(file_wvl.min(), file_wvl.max(), len(file_wvl))

    # Have the computer generate y-values for the spline based on its regression
    file_cs_pred = file_cs.predict(dmatrix(
        "bs(xp, knots=(knot1, knot2, knot3, knot4, knot5, knot6, knot7), include_intercept=False)", {"xp": xp}, return_type='dataframe')
    )

    # Have the computer plot both the original data and the regressed data
    sns.scatterplot(x = file_wvl, y = file_flx)
    plt.plot(xp, file_cs_pred, label='Cubic spline with degree=4 (4 knots)', color='red')
    plt.legend()
    plt.title("Cubic Spline Regression Line for Testing Dataset")
    plt.savefig(OUTPUT_PATH + filename + '_spline.png')
    plt.close()

    # Find the max of the cubic spline data and its corresponding wavelength
    file_index_max3, file_flx_max3 = max(file_cs_pred)
    file_wvl_max3 = file_wvl[file_index_max3]

    # Find temp and store it
    temp3 = temp(file_wvl_max3)

    print("--cubic spline data--")
    print("maximum flux: ..............", file_flx_max3)
    print("corresponging wavelength: ..", file_wvl_max3)
    print("temperature: ...............", temp3)

    return (filename, teff, file_flx_max1, file_wvl_max1, temp1, file_flx_max2, file_wvl_max2, temp2, file_flx_max3, file_wvl_max3, temp3)




In [ ]:
results = pd.DataFrame(columns = ["File", "Teff", "Unalt. Flux", "Unalt. Wvl", "Unalt. Temp", "Conv. Flux", "Conv. Wvl", "Conv. Temp", "Reg. Flux", "Reg. Wvl", "Reg. Temp"])

import os

# Get a list of all files in the folder
file_list = [f for f in os.listdir(DATA_PATH) if os.path.isfile(os.path.join(DATA_PATH, f))]

# Iterate through the files
for filename in file_list:
    print(filename)
    results.loc[len(results.index)] = procedure(filename, 50)
    results.to_csv(OUTPUT_PATH + "results.csv", index = False)

display(results)

In [ ]:
def D(x, y):
    ydiff = np.diff(y)/np.diff(x)
    yprime = []

    for i in range(len(ydiff)):
        if i == 0:
            ytemp = ydiff[i]
        else:
            ytemp = (ydiff[i] + ydiff[i - 1])/2

        yprime = np.append(yprime, ytemp)

    yprime = np.append(yprime, ydiff[len(ydiff) - 1])

    return (yprime)

In [ ]:
def cleaning(width, lstx, lsty):
    lstx_avg = []
    lsty_avg = []

    n = width
    lstx_terms = math.ceil(len(lstx) / n)
    lsty_terms = math.ceil(len(lsty) / n)

    for i in range(lsty_terms):
        sum = 0
        for j in range(n):
            sum += lsty[j + (i * n)]
        avg = sum / n
        lsty_avg.append(avg)

        index = (i * n) + (n / 2)
        lstx_avg.append(lstx[index])

    return lstx_avg, lsty_avg